# Official baselines · FOA-Attack / MPC-Attack / SAGA

N=200 · ε=16 pixel · 300 steps. Author code is locked by commit and SHA-256.

1. Select RTX Pro 6000 in **Configure compute** (Kaggle: enable GPU + Internet).
2. Edit CONFIG: account folder, dataset path/Drive ID, hyperparameters and evaluation models.
3. Run cells, connect Drive using the private OAuth form, optionally enter API keys.
4. Click **Chạy / tiếp tục**. Keep the same RUN_NAME after a session expires.

Checkpoints upload to Drive after each completed attack image. An interrupted image
restarts from the preceding image's saved RNG state. Captions and GPT scores resume
from durable batches. No credentials are stored in notebook code or result archives.

FOA: original fixed K=10, FGSM, source/target crops; MPC: original 300-step MI-FGSM;
SAGA: original released text targets, Qwen3-VL attention and region schedule.
Targets and preprocessing differ by author; details are in README_vi.md.
GPU runtime uses PyTorch 2.8 CUDA 12.8 for Blackwell. Attack precision stays unchanged.
Runs use one method at a time to fit the available 32 GiB system RAM.

Evaluation reuses v3.2 CaptionGenerator and GPTScorer verbatim: LLaVA-1.5-7B,
InternVL3-8B, Qwen3-VL-8B, Gemini 2.5 Flash, GPT-5 mini. The judge is GPT-4o,
temperature=0, max_tokens=100; default ASR uses score >= 0.3. API usage is billed.


In [ ]:
# Edit this cell only when changing account, data, or hyperparameters.
# epsilon is in pixel units [0,255]. Original MPC requires steps=300 and batch=1.
# Use the SAME RUN_NAME/config to resume; choose a NEW RUN_NAME for a new experiment.
# On another account: change drive_parent_id; share/copy the dataset or set data_archive.
from pathlib import Path
import os
CONFIG = {'run_name': 'official_n200_e16_s300_drive_v6',
 'methods': ['foa', 'mpc', 'saga'],
 'n': 200,
 'epsilon': 16,
 'steps': 300,
 'alpha': 1.0,
 'cpu_threads': 4,
 'session_hours': 7.5,
 'data_archive': '',
 'data_drive_file_id': '1V51d_Zk74JY84uJhsUIX-A1y1wfJrDJN',
 'data_sha256': '3518b43ac953db8192a31698105cee4f277cc543351bf28e1eec6d7452987103',
 'backup_drive': True,
 'drive_parent_id': '1daramuahKwdAVGQtsDZTzy1v0OSG-kiI',
 'drive_prefix': 'official_baselines',
 'model_revisions': {'openai/clip-vit-base-patch16': '57c216476eefef5ab752ec549e440a49ae4ae5f3',
                     'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268',
                     'laion/CLIP-ViT-G-14-laion2B-s12B-b42K': '4b0305adc6802b2632e11cbe6606a9bdd43d35c9',
                     'facebook/dinov2-base': 'f9e44c814b77203eaa57a6bdbbd535f21ede1415',
                     'OpenGVLab/InternVL3-1B-hf': '014c0583a0d4bedf29fbe2dbff4f865eb998e171',
                     'Qwen/Qwen3-VL-8B-Instruct': '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b',
                     'llava-hf/llava-1.5-7b-hf': 'b234b804b114d9e37bb655e11cbbb5f5e971b7a9',
                     'OpenGVLab/InternVL3-8B-hf': '259a3b64a14623c0ec91a045cb43f7c5af5fa6af'},
 'mpc': {'lam': 0.6, 'tau': 0.2, 'omega': 2.0},
 'saga': {'seed': 43, 'phases': 10, 'top_k': 3, 'layer': 29},
 'run_evaluation': True,
 'evaluation': {'models': ['LLaVA-1.5-7B', 'InternVL3-8B', 'Qwen3-VL-8B', 'Gemini 2.5 Flash', 'GPT-5 mini'],
                'model_settings': {'LLaVA-1.5-7B': {'provider': 'huggingface',
                                                    'model_id': 'llava-hf/llava-1.5-7b-hf',
                                                    'quantization': 'nf4'},
                                   'InternVL3-8B': {'provider': 'huggingface',
                                                    'model_id': 'OpenGVLab/InternVL3-8B-hf',
                                                    'quantization': 'nf4'},
                                   'Qwen3-VL-8B': {'provider': 'huggingface',
                                                   'model_id': 'Qwen/Qwen3-VL-8B-Instruct',
                                                   'quantization': 'nf4'},
                                   'Gemini 2.5 Flash': {'provider': 'gemini',
                                                        'model_id': 'gemini-2.5-flash',
                                                        'quantization': 'none'},
                                   'GPT-5 mini': {'provider': 'openai',
                                                  'model_id': 'gpt-5-mini-2025-08-07',
                                                  'quantization': 'none'}},
                'prompt': 'Describe this image in one concise sentence, no longer than 20 words.',
                'max_new_tokens': 64,
                'local_batch_size': 8,
                'api_workers': 4,
                'run_judge': True,
                'judge_model': 'gpt-4o',
                'judge_workers': 4,
                'judge_rpm': 12,
                'judge_tpm': 8000,
                'success_threshold': 0.3,
                'success_operator': '>='}}
WORK_ROOT = Path('/kaggle/working' if Path('/kaggle').exists() else '/marimo' if Path('/marimo').exists() else '.') / 'baselines' / CONFIG['run_name']


In [ ]:
# Frozen author sources and orchestration; no dataset or credentials embedded.
import base64 as _b64, hashlib as _hash, io as _io, zipfile as _zip
import subprocess as _sp, sys as _sys, shutil as _shutil, importlib.util as _iu
for _module, _package in [('googleapiclient','google-api-python-client'), ('pydata_google_auth','pydata-google-auth'), ('dotenv','python-dotenv')]:
    if _iu.find_spec(_module) is None:
        _uv = _shutil.which('uv')
        _sp.run([_uv,'pip','install','--python',_sys.executable,_package] if _uv else [_sys.executable,'-m','pip','install',_package], check=True)
_bundle = _b64.b64decode('...')
assert _hash.sha256(_bundle).hexdigest() == '027385e843b8be044f7577befdcab123ef74136f435b2dba890e296e56cd377a'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
with _zip.ZipFile(_io.BytesIO(_bundle)) as _z:
    for _entry in _z.infolist():
        _target = (WORK_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe source archive')
        _data = _z.read(_entry)
        if _target.exists() and _target.read_bytes() != _data: raise ValueError('Run code differs; select a new RUN_NAME')
        _target.parent.mkdir(parents=True, exist_ok=True)
        if not _target.exists(): _target.write_bytes(_data)
if str(WORK_ROOT.resolve()) not in _sys.path: _sys.path.insert(0, str(WORK_ROOT.resolve()))
from baseline_runtime import verify_sources as _verify_sources
_verify_sources(WORK_ROOT)
# Explicit import path for isolated child Python environments.
os.environ['PYTHONPATH'] = str(WORK_ROOT.resolve()) + (os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
from notebook_session import Session as _Session
session = _Session(WORK_ROOT, CONFIG)
_url = session.auth_url() if CONFIG['backup_drive'] else None
if _url:
    from IPython.display import display, Markdown
    display(Markdown('[Connect Drive](' + _url + ')'))
    import getpass
    session.connect(getpass.getpass('OAuth callback URL: '))
for _key in ('OPENAI_API_KEY','GEMINI_API_KEY'):
    if CONFIG['run_evaluation'] and not os.environ.get(_key):
        import getpass
        _value=getpass.getpass(_key+' (empty to defer): ')
        if _value: session.set_secret(_key,_value)


In [ ]:
session.launch()


In [ ]:
session.status()  # Re-run this cell for current progress.
